In [1]:
# https://www.tensorflow.org/api_docs/python/tf/keras/layers

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re
import nltk
import emoji
import spacy
import string
import tensorflow as tf
import unicodedata
import datetime
from sklearn.utils import shuffle
from joblib import dump, load
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import CSVLogger, TensorBoard, EarlyStopping
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split

In [2]:
# transformando e abrindo o arquivo em formato csv
df = pd.read_excel('DateToTestSentiment_20210817.xls.xls')

In [3]:
def preprocess_data(data, columns,
                    null=True):
    
    df = data[columns]
    
    if null:
        df = df.dropna().reset_index().drop(columns=['index'])
    
    return df

def preprocess_text(text, 
                    remove_stop = True, 
                    stem_words = False, 
                    remove_mentions_hashtags = True
                   ):
    """
    eg:
    input: preprocess_text("@water #dream hi hello where are you going be there tomorrow happening happen happens",  
    stem_words = True) 
    output: ['tomorrow', 'happen', 'go', 'hello']
    """

    # Remove emojis
    emoji_pattern = re.compile("[" "\U0001F1E0-\U0001F6FF" "]+", flags=re.UNICODE)
    text = emoji_pattern.sub(r"", text)
    text = "".join([x for x in text if x not in emoji.UNICODE_EMOJI])
    
    # Corrige o bug que elimina parte das palavras com acentos
    text = ''.join(ch for ch in unicodedata.normalize('NFKD', text) 
    if not unicodedata.combining(ch))

    if remove_mentions_hashtags:
        text = re.sub(r"@(\w+)", " ", text)
        text = re.sub(r"#(\w+)", " ", text)

    text = re.sub(r"[^\x00-\x7F]+", " ", text)
    regex = re.compile('[' + re.escape(string.punctuation) + '0-9\\r\\t\\n]')
    nopunct = regex.sub(" ", text.lower())
    words = (''.join(nopunct)).split()

    if(remove_stop):
        words = [w for w in words if w not in portuguese_stopwords]
        words = [w for w in words if len(w) > 2]  

    if(stem_words):
        stemmer = PorterStemmer()
        words = [stemmer.stem(w) for w in words]

    return list(words)


def tokenizacao(df):
    
    # cria a coluna com textos vetorizados
    rows, cols = df.shape

    df['token'] = [preprocess_text(df["COMMENT_TEXT"][row]) for row in range(rows)]
    
    return df


def treinar_vetorizacao(df):

    # coleciona as palavras usadas para o treinamento da função CountVectorizer
    lista_treino = []

    for item in df['token']:
        lista_treino1 = [n for n in item if n not in lista_treino]
        lista_treino.extend(lista_treino1)

    # treina o modelo de vetorização
    vectorize = CountVectorizer(lowercase=True, strip_accents = 'unicode')

    vectorize.fit(lista_treino)
    
    return vectorize, lista_treino


def vectorize2(lista):
    lista2 = [vectorize.vocabulary_[item] for item in lista]
    
    return lista2

In [ ]:
portuguese_stopwords = nltk.corpus.stopwords.words('portuguese')

#
df2 = preprocess_data(df, 
                      columns=['KEY','COMMENT_ID','COMMENT_TEXT'],
                      null = True
                     )

#
df2 = tokenizacao(df2)

#
vectorize, lista_treino = treinar_vetorizacao(df2)

# criando a coluna com o texto vetorizado
df2['vectors'] = df2['token'].apply(vectorize2)

In [5]:
df['RATING_VALUE'].unique()

array(['5', '4', '3', '1', '2', nan, '5,0 ', '3,0 ', '4,0 ', '1,0 ',
       '2,0 '], dtype=object)

In [ ]:
df2

In [ ]:
topics = ['AQUECIMENTO', 'ASSISTÊNCIA TÉCNICA', 'ATENDIMENTO', 'AUTO FALANTE', 'BATERIA', 'CAMERA', 
         'CARREGADOR', 'CUSTO BENEFICIO', 'DESIGN', 'ENTREGA', 'FLASH', 'FONE', 'JOGOS', 'MEMÓRIA',
         'PESO', 'PREÇO', 'PROCESSADOR', 'QUALIDADE', 'RESISTÊNCIA', 'TAMANHO', 'TELA', 'TRAVAMENTO',
         'VELOCIDADE', 'GENÉRICO/OUTRO']

In [ ]:
len(topics)

In [ ]:
topics_token = [preprocess_text(item) for item in topics]

In [ ]:
topics_token

In [ ]:
# criando clusters

lista_clusters = []

for n, objeto in enumerate(topics_token):
    if len(objeto) == 1:
        lista_1 = [item for item in df2.token if objeto[0]
                   in item and (len(item) > 20 and len(item) < 30)]
        lista_clusters.append(lista_1)

    else:
        lista_2 = [item for item in df2.token if (
            objeto[0] in item) and (objeto[1] in item) and (len(item) > 20 and len(item) < 30)]
        lista_clusters.append(lista_2)

In [ ]:
len(lista_clusters)

In [ ]:
lista_clusters[0]

In [ ]:
[len(item) for item in lista_clusters]

In [ ]:
# criando centros dos clusters (exceto do ultimo)

centro_clusters = [item[0] for item in lista_clusters[0:23]]

In [ ]:
centro_clusters

In [ ]:
# criando último cluster

lista_clusters1 = []

for n, objeto in enumerate(topics_token):
    if len(objeto) == 1:
        lista_1 = [item for item in df2.token if objeto[0] in item]
        lista_clusters1.extend(lista_1)

    else:
        lista_2 = [item for item in df2.token if (
            objeto[0] in item) and (objeto[1] in item)]
        lista_clusters1.extend(lista_2)

ultimo_cluster = [item for item in df2.token if (
    item not in lista_clusters1) and (len(item) > 20 and len(item) < 30)]

In [ ]:
ultimo_cluster

In [ ]:
np.random.randint(20)

In [ ]:
# criando ultimo centro dos clusters

x = np.random.randint(len(ultimo_cluster))

ultimo_centro = ultimo_cluster[x]

In [ ]:
# juntando tudo

centro_clusters.append(ultimo_centro)

In [ ]:
len(centro_clusters)